1. import + env

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Notebook location: api/notebooks
API_ROOT = Path("..").resolve()                  # .../ai-tv-pro/api
TV_DATA_DIR = (API_ROOT / "tv_data").resolve()   # .../ai-tv-pro/api/tv_data

# Load backend env (isti kao agent.py)
load_dotenv(API_ROOT / ".env")

# Allow imports from api/
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))

print("API_ROOT:", API_ROOT)
print("TV_DATA_DIR:", TV_DATA_DIR)
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))

from app.agent import tv_rag_search, advanced_retrieve, get_duckdb

print("Imported app.agent OK")
print("Has advanced_retrieve:", advanced_retrieve is not None)

API_ROOT: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api
TV_DATA_DIR: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data
OPENAI_API_KEY present: True


c:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\.venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imported app.agent OK
Has advanced_retrieve: True


2. testset and structure

In [2]:
import json
from pathlib import Path

TESTSET_PATH = API_ROOT / "tv_data" / "testset_tv.json"

print("Looking at:", TESTSET_PATH)
print("Exists:", TESTSET_PATH.exists())

data = json.loads(TESTSET_PATH.read_text(encoding="utf-8"))

print("Loaded rows:", len(data))
print("Type:", type(data))
print("Keys in first row:", list(data[0].keys()))

Looking at: C:\Users\gpudja\OneDrive - Hrvatski Telekom\00_Posao\RAZNO\99_EDUKACIJE\16_AI_Eng_Bootcamp\certification_challenge\ai-tv-pro\api\tv_data\testset_tv.json
Exists: True
Loaded rows: 27
Type: <class 'list'>
Keys in first row: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


3. normalizing testset

In [3]:
def get_question(row: dict) -> str:
    return row["user_input"]

def get_reference(row: dict) -> str:
    return row["reference"]

def get_reference_contexts(row: dict) -> list[str]:
    return row.get("reference_contexts", []) or []

# quick sanity check
print("Sample question:\n", get_question(data[0]))
print("\nReference answer preview:\n", get_reference(data[0])[:200])
print("\n# reference contexts:", len(get_reference_contexts(data[0])))

Sample question:
 In the context of the IPTV Monthly Report for December 2024, how does the performance of HRT 1 in terms of LTV compare to other channels based on total minutes and unique viewers?

Reference answer preview:
 In the IPTV Monthly Report for December 2024, HRT 1 is the top entry with 33,726,930 minutes of playback and 4,972,636 unique viewers, indicating a strong performance in terms of LTV compared to other

# reference contexts: 1


4. Retrived context (Baseline vs Improved)

In [4]:
import asyncio

def retrieve_basic(q: str) -> list[str]:
    txt = tv_rag_search.invoke({"query": q})
    if not txt or "No relevant internal context" in txt:
        return []
    return [txt]

async def retrieve_advanced(q: str) -> list[str]:
    txt = await advanced_retrieve.ainvoke({"query": q})
    if not txt or "No relevant internal context" in txt:
        return []
    parts = [p.strip() for p in txt.split("\n\n") if p.strip()]
    return parts

# quick sanity check on first row
q0 = get_question(data[0])

basic0 = retrieve_basic(q0)
adv0 = await retrieve_advanced(q0)   # <-- OVO JE BITNO

print("Q0:", q0[:120], "...")
print("\nBASIC contexts:", len(basic0))
print("ADV contexts:", len(adv0))

print("\n--- BASIC preview ---")
print((basic0[0][:500] + "...") if basic0 else "NO_CTX")

print("\n--- ADV preview ---")
print((adv0[0][:500] + "...") if adv0 else "NO_CTX")

Q0: In the context of the IPTV Monthly Report for December 2024, how does the performance of HRT 1 in terms of LTV compare t ...

BASIC contexts: 1
ADV contexts: 3

--- BASIC preview ---
- month_partition=202512 | watch_date=2025-12-31 | channelname=HRT 4 | content_playback_type=LTV | daily_total_minute=3538220,62 | nr_unique_viewers=50215 (meta={'row': 127451, 'file': 'Monthly iptv - channel monthly rating.csv', '_id': '317b3a1b-1d67-464a-9b3c-ed0317611d37', '_collection_name': 'tv_monthly_channel_rating'})
- month_partition=202512 | watch_date=2025-12-31 | channelname=HRT 3 | content_playback_type=LTV | daily_total_minute=2957636,6 | nr_unique_viewers=49983 (meta={'row': 12176...

--- ADV preview ---
month_partition=202512 | watch_date=2025-12-31 | channelname=HRT 4 | content_playback_type=LTV | daily_total_minute=3538220,62 | nr_unique_viewers=50215...


5. Retrieve Contexts for all questions

In [5]:
basic_contexts = []
adv_contexts = []

# baseline (sync)
for row in data:
    q = get_question(row)
    basic_contexts.append(retrieve_basic(q))

# improved (async) — use await in a loop
for row in data:
    q = get_question(row)
    adv_contexts.append(await retrieve_advanced(q))

print("Total questions:", len(data))
print("Basic empty contexts:", sum(1 for x in basic_contexts if not x))
print("Adv empty contexts:", sum(1 for x in adv_contexts if not x))

# quick spot check random-like: first 3 lengths
print("Basic ctx lens (first 3):", [len(x) for x in basic_contexts[:3]])
print("Adv ctx lens (first 3):", [len(x) for x in adv_contexts[:3]])

Total questions: 27
Basic empty contexts: 0
Adv empty contexts: 0
Basic ctx lens (first 3): [1, 1, 1]
Adv ctx lens (first 3): [3, 3, 3]


6. Generate Answers from Retrieved Context

In [6]:
from langchain_openai import ChatOpenAI

GEN_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
gen_llm = ChatOpenAI(model=GEN_MODEL, temperature=0.0)

def answer_from_context(question: str, contexts: list[str]) -> str:
    ctx_block = "\n\n".join(contexts[:3]) if contexts else ""
    prompt = (
        "You are BroadcastIQ.\n"
        "Answer the question using ONLY the provided context.\n"
        'If the context does not contain the answer, respond exactly: "Insufficient internal context."\n\n'
        f"QUESTION:\n{question}\n\n"
        f"CONTEXT:\n{ctx_block}\n"
    )
    return gen_llm.invoke(prompt).content

# quick sanity check on first question (baseline vs improved)
q0 = get_question(data[0])
ref0 = get_reference(data[0])

resp_basic_0 = answer_from_context(q0, basic_contexts[0])
resp_adv_0   = answer_from_context(q0, adv_contexts[0])

print("Q0:", q0[:100], "...")
print("\nREF preview:", ref0[:180], "...")
print("\nBASIC resp preview:", resp_basic_0[:220], "...")
print("\nADV resp preview:", resp_adv_0[:220], "...")

Q0: In the context of the IPTV Monthly Report for December 2024, how does the performance of HRT 1 in te ...

REF preview: In the IPTV Monthly Report for December 2024, HRT 1 is the top entry with 33,726,930 minutes of playback and 4,972,636 unique viewers, indicating a strong performance in terms of L ...

BASIC resp preview: Based on the provided December data (month_partition=202512), HRT 1 strongly outperforms the other listed channels in LTV:

- HRT 1 (three records): total minutes ~28.05M, 29.40M, and 31.36M with unique viewers 151,044;  ...

ADV resp preview: Insufficient internal context. ...


7. Retrieval Query Shortening (Evaluation-only)

In [7]:
import re

def short_retrieval_query(q: str) -> str:
    ql = q.lower()

    # 1) entity / channel hints
    channel_match = re.findall(r"(hrt\s*\d+|doma\s*tv|nova\s*tv|rtl\s*\d*|hbo\s*\w*)", ql)
    channel_part = " ".join(dict.fromkeys(channel_match)) if channel_match else ""

    # 2) playback type hints
    pb = []
    if "ltv" in ql: pb.append("LTV")
    if "vod" in ql: pb.append("VOD")
    if "live" in ql: pb.append("LIVE")

    # 3) time hint (e.g., December 2024)
    time_part = ""
    m = re.search(
        r"(january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4}",
        ql
    )
    if m:
        time_part = m.group(0)

    # 4) de-duplicated metrics (prefer the more specific phrase)
    metrics = []
    if "total minutes" in ql:
        metrics.append("total minutes")
    elif "minutes" in ql:
        metrics.append("minutes")

    if "unique viewers" in ql:
        metrics.append("unique viewers")
    elif "viewers" in ql:
        metrics.append("viewers")

    parts = [p for p in [channel_part, " ".join(pb), time_part, " ".join(metrics)] if p]
    return " ".join(parts) if parts else q

# show example for Q0
print("Original Q0:\n", q0)
print("\nShort Q0:\n", short_retrieval_query(q0))

Original Q0:
 In the context of the IPTV Monthly Report for December 2024, how does the performance of HRT 1 in terms of LTV compare to other channels based on total minutes and unique viewers?

Short Q0:
 hrt 1 LTV december 2024 total minutes unique viewers


8. Sanity Check: Retrieval Using Short Query 

In [8]:
def retrieve_basic_short(q: str) -> list[str]:
    rq = short_retrieval_query(q)
    txt = tv_rag_search.invoke({"query": rq})
    if not txt or "No relevant internal context" in txt:
        return []
    return [txt]

async def retrieve_advanced_short(q: str) -> list[str]:
    rq = short_retrieval_query(q)
    txt = await advanced_retrieve.ainvoke({"query": rq})
    if not txt or "No relevant internal context" in txt:
        return []
    parts = [p.strip() for p in txt.split("\n\n") if p.strip()]
    return parts

# Q0 sanity check
q0 = get_question(data[0])

basic0_s = retrieve_basic_short(q0)
adv0_s = await retrieve_advanced_short(q0)

print("Short query:", short_retrieval_query(q0))

print("\nBASIC(short) contexts:", len(basic0_s))
print("ADV(short) contexts:", len(adv0_s))

print("\n--- BASIC(short) preview ---")
print((basic0_s[0][:600] + "...") if basic0_s else "NO_CTX")

print("\n--- ADV(short) preview ---")
print((adv0_s[0][:600] + "...") if adv0_s else "NO_CTX")

# quick check: does the preview contain "channelname=HRT 1" ?
print("\nContains HRT 1 in BASIC(short)?", ("channelname=HRT 1" in basic0_s[0]) if basic0_s else False)
print("Contains HRT 1 in ADV(short)?", any("channelname=HRT 1" in c for c in adv0_s))

Short query: hrt 1 LTV december 2024 total minutes unique viewers

BASIC(short) contexts: 1
ADV(short) contexts: 3

--- BASIC(short) preview ---
- month_partition=202412 | watch_date=2024-12-04 | channelname=HRT 4 | content_playback_type=LTV | daily_total_minute=4465932,21 | nr_unique_viewers=58690 (meta={'row': 18458, 'file': 'Monthly iptv - channel monthly rating.csv', '_id': 'dcac17d6-087a-4068-b369-7601c74f412f', '_collection_name': 'tv_monthly_channel_rating'})
- month_partition=202412 | watch_date=2024-12-10 | channelname=HRT 1 | content_playback_type=LTV | daily_total_minute=33726930 | nr_unique_viewers=164042 (meta={'row': 16103, 'file': 'Monthly iptv - channel monthly rating.csv', '_id': '7a4c4b13-6dba-40c1-ab4a-b12ca6f6a821',...

--- ADV(short) preview ---
month_partition=202412 | watch_date=2024-12-04 | channelname=HRT 4 | content_playback_type=LTV | daily_total_minute=4465932,21 | nr_unique_viewers=58690...

Contains HRT 1 in BASIC(short)? True
Contains HRT 1 in ADV(short)?

9. Retrieve Contexts for All Questions (Using Short Queries)

In [9]:
basic_contexts_s = []
adv_contexts_s = []

# baseline (sync)
for row in data:
    q = get_question(row)
    basic_contexts_s.append(retrieve_basic_short(q))

# improved (async)
for row in data:
    q = get_question(row)
    adv_contexts_s.append(await retrieve_advanced_short(q))

print("Total questions:", len(data))
print("Basic(short) empty contexts:", sum(1 for x in basic_contexts_s if not x))
print("Adv(short) empty contexts:", sum(1 for x in adv_contexts_s if not x))

print("Basic(short) ctx lens (first 3):", [len(x) for x in basic_contexts_s[:3]])
print("Adv(short) ctx lens (first 3):", [len(x) for x in adv_contexts_s[:3]])

Total questions: 27
Basic(short) empty contexts: 0
Adv(short) empty contexts: 0
Basic(short) ctx lens (first 3): [1, 1, 1]
Adv(short) ctx lens (first 3): [3, 3, 3]


10. Generate Answers for All Questions (Short-Query Contexts)

In [10]:
answers_basic_s = []
answers_adv_s = []

for i, row in enumerate(data):
    q = get_question(row)
    answers_basic_s.append(answer_from_context(q, basic_contexts_s[i]))
    answers_adv_s.append(answer_from_context(q, adv_contexts_s[i]))

print("Generated answers:", len(answers_basic_s), len(answers_adv_s))

# quick sanity check on Q0 again
print("\nQ0:", get_question(data[0])[:100], "...")
print("\nBASIC(short) answer preview:", answers_basic_s[0][:220], "...")
print("\nADV(short) answer preview:", answers_adv_s[0][:220], "...")

Generated answers: 27 27

Q0: In the context of the IPTV Monthly Report for December 2024, how does the performance of HRT 1 in te ...

BASIC(short) answer preview: Based on the December 2024 entries in the report, HRT 1’s LTV performance is substantially higher than the other channel shown (HRT 4):

- HRT 1 (Dec 2024): total minutes ≈ 33.7M–35.54M; unique viewers ≈ 164,042–164,849. ...

ADV(short) answer preview: HRT 1 strongly outperforms the other channel(s) in the provided December 2024 LTV data. On 2024-12-04 HRT 1 logged 35,541,895.78 total minutes and 164,849 unique viewers (and on 2024-12-10 33,726,930 minutes and 164,042  ...


11. RAGAS Evaluation Datasets (Baseline vs Improved)

In [11]:
baseline_rows_s = []
improved_rows_s = []

for i, row in enumerate(data):
    q = get_question(row)
    ref = get_reference(row)

    baseline_rows_s.append({
        "user_input": q,
        "retrieved_contexts": basic_contexts_s[i],
        "response": answers_basic_s[i],
        "reference": ref,
    })

    improved_rows_s.append({
        "user_input": q,
        "retrieved_contexts": adv_contexts_s[i],
        "response": answers_adv_s[i],
        "reference": ref,
    })

print("Baseline rows:", len(baseline_rows_s))
print("Improved rows:", len(improved_rows_s))

# sanity: show keys
print("Keys:", baseline_rows_s[0].keys())

Baseline rows: 27
Improved rows: 27
Keys: dict_keys(['user_input', 'retrieved_contexts', 'response', 'reference'])


12. RAGAS evaluation -> Baseline: tv_rag_search

In [12]:
import ragas, inspect
from ragas import evaluate
print("ragas version:", getattr(ragas, "__version__", "unknown"))
print("evaluate signature:", inspect.signature(evaluate))

ragas version: 0.4.3
evaluate signature: (dataset: 't.Union[Dataset, EvaluationDataset]', metrics: 't.Optional[t.Sequence[Metric]]' = None, llm: 't.Optional[BaseRagasLLM | LangchainLLM]' = None, embeddings: 't.Optional[BaseRagasEmbeddings | BaseRagasEmbedding | LangchainEmbeddings]' = None, experiment_name: 't.Optional[str]' = None, callbacks: 'Callbacks' = None, run_config: 't.Optional[RunConfig]' = None, token_usage_parser: 't.Optional[TokenUsageParser]' = None, raise_exceptions: 'bool' = False, column_map: 't.Optional[t.Dict[str, str]]' = None, show_progress: 'bool' = True, batch_size: 't.Optional[int]' = None, _run_id: 't.Optional[UUID]' = None, _pbar: 't.Optional[tqdm]' = None, return_executor: 'bool' = False, allow_nest_asyncio: 'bool' = True) -> 't.Union[EvaluationResult, Executor]'


In [16]:
from ragas import EvaluationDataset, evaluate
from ragas.llms import llm_factory
from ragas.run_config import RunConfig

# metrics (tocni pathovi za ragas 0.4.3)
from ragas.metrics._context_recall import LLMContextRecall
from ragas.metrics._faithfulness import Faithfulness
from ragas.metrics._factual_correctness import FactualCorrectness

from openai import OpenAI
import os

# 1) OpenAI client s vecim timeoutom + retry
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=180.0,
    max_retries=6,
)

# 2) evaluator llm
evaluator_llm = llm_factory("gpt-4o-mini", client=client)

# 3) run config: minimalna paralela (najstabilnije)
run_config = RunConfig(
    timeout=180,        # timeout po jobu u sekundama
    max_retries=6,
    max_wait=60,
    max_workers=1       # KRUCIALNO: bez paralelizma
)

print("OK imports + evaluator_llm + run_config ready")

OK imports + evaluator_llm + run_config ready


In [17]:
import inspect

metrics = [
    LLMContextRecall(),
    Faithfulness(),
    FactualCorrectness(mode="f1"),
]

# Sigurno postavi llm na svaki metric ako podrzava
for m in metrics:
    if hasattr(m, "set_llm") and callable(getattr(m, "set_llm")):
        try:
            m.set_llm(evaluator_llm)
            continue
        except Exception as e:
            print(f"set_llm failed for {m.__class__.__name__}: {e}")

    if hasattr(m, "llm"):
        try:
            m.llm = evaluator_llm
        except Exception as e:
            print(f"llm assign failed for {m.__class__.__name__}: {e}")

# ispisi kratki status
for m in metrics:
    has_llm = hasattr(m, "llm") and getattr(m, "llm") is not None
    print(m.__class__.__name__, "llm_set=" + str(has_llm))

LLMContextRecall llm_set=True
Faithfulness llm_set=True
FactualCorrectness llm_set=True


In [18]:
# build dataset
baseline_dataset_s = EvaluationDataset.from_list(baseline_rows_s)
print("rows:", len(baseline_rows_s), "expected jobs:", len(baseline_rows_s) * len(metrics))

baseline_result_s = evaluate(
    dataset=baseline_dataset_s,
    metrics=metrics,
    llm=evaluator_llm,
    run_config=run_config,
    raise_exceptions=False,
    show_progress=True,
    batch_size=1,
)

baseline_df_s = baseline_result_s.to_pandas()
baseline_df_s.head()

rows: 27 expected jobs: 81


Evaluating:  53%|█████▎    | 43/81 [08:41<05:58,  9.43s/it]Exception raised in Job[43]: InstructorRetryException(<failed_attempts>

<generation number="1">
<exception>
    The output is incomplete due to a max_tokens length limit.
</exception>
<completion>
    ChatCompletion(id='chatcmpl-DEak9dqkIT3sCcwu1cO4VaePKNEVL', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n    "statements": [\n        {\n            "statement": "The channel is HRT 3.",\n            "reason": "The context explicitly mentions \'channelname=HRT 3\'.",\n            "verdict": 1\n        },\n        {\n            "statement": "The report is the IPTV Monthly Report for May 2025.",\n            "reason": "The context includes data for the month of May 2025, indicating it is a monthly report.",\n            "verdict": 1\n        },\n        {\n            "statement": "On May 28, 2025, the content playback type was IR.",\n            "reason": "The context s

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,factual_correctness(mode=f1)
0,In the context of the IPTV Monthly Report for ...,[- month_partition=202412 | watch_date=2024-12...,Based on the December 2024 entries in the repo...,"In the IPTV Monthly Report for December 2024, ...",0.0,1.000000,0.0
1,What are the viewer statistics for Doma TV ove...,[- month_partition=202503 | watch_date=2025-03...,Viewer statistics for Doma TV (last 31 days) f...,"Doma TV had a total of 6,326,153 minutes of pl...",0.0,0.857143,0.0
2,What top channel in IPTV Monthly Report?,[- month_partition=202512 | watch_date=2025-12...,ICTbusiness TV,Top entry is Nova TV (LTV) with 51622038 minut...,0.0,0.000000,0.0
3,How does RTL perform in terms of viewer engage...,[- month_partition=202504 | watch_date=2025-04...,"Based on the provided rows, RTL Living shows s...","RTL has a total of 4,999,005 viewers and 17,99...",0.0,0.571429,0.0
4,What is the LTV and how many minutes and viewe...,[- month_partition=202512 | watch_date=2025-12...,Insufficient internal context.,The LTV for MAXSport 1 is 14939709 minutes and...,0.0,0.000000,0.0


13. Baseline summary

In [20]:
import pandas as pd

baseline_df_s = baseline_result_s.to_pandas()

# tocne metric kolone
metric_cols = [
    "context_recall",
    "faithfulness",
    "factual_correctness(mode=f1)"
]

summary = {}

for c in metric_cols:
    s = pd.to_numeric(baseline_df_s[c], errors="coerce")
    summary[c + "_mean"] = float(s.mean())
    summary[c + "_nan"] = int(s.isna().sum())

# success rate = redovi gdje nijedna metric nije NaN
metric_numeric = baseline_df_s[metric_cols].apply(pd.to_numeric, errors="coerce")
summary["success_rate"] = float(1 - metric_numeric.isna().any(axis=1).mean())
summary["n_rows"] = len(baseline_df_s)

summary

{'context_recall_mean': 0.0,
 'context_recall_nan': 0,
 'faithfulness_mean': 0.15109890109890112,
 'faithfulness_nan': 1,
 'factual_correctness(mode=f1)_mean': 0.0,
 'factual_correctness(mode=f1)_nan': 0,
 'success_rate': 0.962962962962963,
 'n_rows': 27}

14. Improved evaluation (Advanced Retriever)

In [22]:
# build dataset (improved)
improved_dataset_s = EvaluationDataset.from_list(improved_rows_s)
print("rows:", len(improved_rows_s), "expected jobs:", len(improved_rows_s) * len(metrics))

improved_result_s = evaluate(
    dataset=improved_dataset_s,
    metrics=metrics,
    llm=evaluator_llm,
    run_config=run_config,
    raise_exceptions=False,
    show_progress=True,
    batch_size=1,
)

improved_df_s = improved_result_s.to_pandas()
improved_df_s.head()

rows: 27 expected jobs: 81


Evaluating: 100%|██████████| 81/81 [13:05<00:00,  9.70s/it]


,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,factual_correctness(mode=f1)
0,In the context of the IPTV Monthly Report for ...,[month_partition=202412 | watch_date=2024-12-0...,HRT 1 strongly outperforms the other channel(s...,"In the IPTV Monthly Report for December 2024, ...",0.0,0.777778,0.22
1,What are the viewer statistics for Doma TV ove...,[month_partition=202503 | watch_date=2025-03-2...,month_partition=202503 | watch_date=2025-03-27...,"Doma TV had a total of 6,326,153 minutes of pl...",0.0,1.000000,0.00
2,What top channel in IPTV Monthly Report?,[month_partition=202512 | watch_date=2025-12-3...,ICTbusiness TV,Top entry is Nova TV (LTV) with 51622038 minut...,0.0,0.000000,0.00
3,How does RTL perform in terms of viewer engage...,[month_partition=202512 | watch_date=2025-12-1...,Insufficient internal context.,"RTL has a total of 4,999,005 viewers and 17,99...",0.0,0.000000,0.00
4,What is the LTV and how many minutes and viewe...,[month_partition=202512 | watch_date=2025-12-1...,Insufficient internal context.,The LTV for MAXSport 1 is 14939709 minutes and...,0.0,0.000000,0.00


15. Improved summary

In [23]:
import pandas as pd

metric_cols = [
    "context_recall",
    "faithfulness",
    "factual_correctness(mode=f1)"
]

summary_improved = {}

for c in metric_cols:
    s = pd.to_numeric(improved_df_s[c], errors="coerce")
    summary_improved[c + "_mean"] = float(s.mean())
    summary_improved[c + "_nan"] = int(s.isna().sum())

metric_numeric = improved_df_s[metric_cols].apply(pd.to_numeric, errors="coerce")
summary_improved["success_rate"] = float(1 - metric_numeric.isna().any(axis=1).mean())
summary_improved["n_rows"] = len(improved_df_s)

summary_improved

{'context_recall_mean': 0.0,
 'context_recall_nan': 0,
 'faithfulness_mean': 0.13991769547325103,
 'faithfulness_nan': 0,
 'factual_correctness(mode=f1)_mean': 0.016296296296296295,
 'factual_correctness(mode=f1)_nan': 0,
 'success_rate': 1.0,
 'n_rows': 27}

16. Comparison table (baseline vs improved)

In [24]:
import pandas as pd

comparison = pd.DataFrame([
    {"variant": "baseline", **summary},
    {"variant": "improved", **summary_improved},
])

comparison

,variant,context_recall_mean,context_recall_nan,faithfulness_mean,faithfulness_nan,factual_correctness(mode=f1)_mean,factual_correctness(mode=f1)_nan,success_rate,n_rows
0,baseline,0.0,0,0.151099,1,0.000000,0,0.962963,27
1,improved,0.0,0,0.139918,0,0.016296,0,1.000000,27


## Retriever Upgrade Evaluation (Baseline vs Improved)

### Baseline vs Advanced Retriever (Reranking)

- Context Recall: **0.000 → 0.000**
- Faithfulness: **0.151 → 0.140**
- Factual Correctness (F1): **0.000 → 0.016**
- Success Rate: **96% → 100%**

### Interpretation

The advanced retriever (semantic retrieval + reranking) did not produce significant improvements on this evaluation dataset.

- **Context Recall remains 0.0** in both cases, indicating that the retrieved contexts do not sufficiently overlap with the reference contexts expected by RAGAS.
- **Faithfulness slightly decreased** (0.151 → 0.140), suggesting that reranking did not meaningfully improve grounding of the generated responses.
- **Factual Correctness improved slightly** (0.000 → 0.016), indicating a small semantic alignment improvement.
- **Evaluation stability improved**, as the advanced retriever achieved a 100% success rate with no token-length failures.

### Engineering Insight

The limited improvement suggests that:

1. The dataset may not be complex enough for reranking to show a strong advantage.
2. Retrieval quality may be constrained by the current chunking strategy.
3. A hybrid approach (e.g., BM25 + dense retrieval + reranking) may yield stronger improvements.
4. Ground-truth formatting may not be fully aligned with how RAGAS evaluates context recall.

Further improvements would likely require:
- Enhanced chunking strategy
- Better alignment between ground truth and retrieved contexts
- Hybrid retrieval architecture
- Query rewriting or structured retrieval


## SQL Correctness Note

The SQL correctness evaluation is identical between baseline and improved retriever variants.

This is expected because SQL-style questions are executed deterministically via the `tv_sql` tool, and are not affected by retrieval strategy or reranking.

Therefore, SQL correctness results are reported once in the baseline evaluation notebook.